# LIFT Pipeline v2: Raw Dataset → Serialization → LLM Inference
**Based on:** Dinh et al. (2022) — *LIFT: Language-Interfaced Fine-Tuning for Non-Language Machine Learning Tasks*

### Fixes & Improvements (v2)
1. **Label normalization** — trailing dot stripped (`>50K.` → `>50K`), asserts exactly 2 classes
2. **Dropped non-semantic features** — `fnlwgt` and `education-num` excluded from serialization
3. **Instruction-tuned model** — default: `Llama-3.2-3B-Instruct`
4. **4-bit quantization enabled by default** on GPU + `batch_size=8`
5. **Sample size raised** to 500 (configurable)
6. **`max_new_tokens` reduced** to 10 for direct mode (only label needed)
7. **Full evaluation** — F1, Precision, Recall, ROC-AUC, confusion matrix
8. **Ablation loop** over all 3 serialization formats (lift / list / compact) × Direct / CoT
9. **CoT-aware parser** with `Final Answer:` marker extraction
10. **Perturbation-based XAI** — leave-one-out feature importance
11. **CoT Faithfulness metric** — Overlap@K vs. perturbation importance
12. **All runs logged to MLflow** with consistent naming

## 0. Install Dependencies

In [1]:
%pip install -q transformers accelerate datasets pandas scikit-learn ucimlrepo mlflow seaborn
%pip install -q -U bitsandbytes>=0.46.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 67.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 85.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 42.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 907.5/907.5 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 10.9 MB/s eta 0:00:00
Note: you may need to res

## 1. Imports & Config
Change `CONFIG` to switch models, formats, and CoT mode.

In [3]:
import os, time, re, warnings
import numpy as np
import pandas as pd
from typing import Optional

from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

warnings.filterwarnings("ignore")

# ── CONFIG ─────────────────────────────────────────────────────────────────────
# CPU-feasible   : "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "microsoft/phi-2"
# Recommended    : "meta-llama/Llama-3.2-3B-Instruct"  (instruction-tuned, T4 GPU)
# Larger (4-bit) : "meta-llama/Meta-Llama-3-8B-Instruct"
CONFIG = {
    "model_id"             : "Qwen/Qwen2.5-3B-Instruct",
    "task_prompt"          : "Does this person earn more than 50K per year? Answer with '>50K' or '<=50K' only.",
    "serial_format"        : "lift",               # lift | list | compact
    "use_cot"              : False,
    "device"               : "cuda" if torch.cuda.is_available() else "cpu",
    "use_4bit"             : torch.cuda.is_available(),  # auto-enable on GPU
    "n_inference_samples"  : 500,                  # raise to 1000 for final thesis run
    "max_new_tokens_direct": 10,                   # direct: only label token needed
    "max_new_tokens_cot"   : 200,                  # cot: needs reasoning space
    "batch_size"           : 8,                    # reduce to 4 if OOM on T4
}

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print(f"Device  : {CONFIG['device']}")
print(f"4-bit   : {CONFIG['use_4bit']}")
print(f"Model   : {CONFIG['model_id']}")

Device  : cuda
4-bit   : True
Model   : Qwen/Qwen2.5-3B-Instruct


## 2. Load & Split Dataset
**FIX #1**: Normalize labels — strip trailing dot from UCI test split labels (`>50K.` → `>50K`).  
**FIX #2**: Drop `fnlwgt` (non-semantic census weight) and `education-num` (redundant with `education`).

In [4]:
def normalize_label(lbl: str) -> str:
    """Strip trailing dot added in UCI Adult test split ('>50K.' -> '>50K')."""
    return str(lbl).strip().rstrip('.')


def load_adult_dataset(drop_cols=("fnlwgt", "education-num")):
    """
    Load UCI Adult dataset with label normalization and feature cleanup.
    - Strips trailing dot from target labels (UCI quirk)
    - Drops fnlwgt (census weight) and education-num (redundant with 'education')
    Returns: X (features), y (normalized labels), feature_names (list)
    """
    print("Loading Adult dataset from UCI ML Repo...")
    dataset = fetch_ucirepo(id=2)
    X = dataset.data.features.copy()
    y_raw = dataset.data.targets.iloc[:, 0].astype(str)

    # Normalize labels
    y = y_raw.apply(normalize_label)
    assert y.nunique() == 2, (
        f"Expected 2 classes after normalization, got: {y.value_counts().to_dict()}"
    )

    # Drop non-semantic columns
    cols_to_drop = [c for c in drop_cols if c in X.columns]
    X = X.drop(columns=cols_to_drop)
    print(f"Dropped columns  : {cols_to_drop}")
    print(f"Shape            : {X.shape}")
    print(f"Class distribution: {y.value_counts().to_dict()}")
    return X, y, X.columns.tolist()


X, y, feature_names = load_adult_dataset()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"\nTrain: {X_train.shape} | Test: {X_test.shape}")
print(f"Feature names: {feature_names}")

Loading Adult dataset from UCI ML Repo...
Dropped columns  : ['fnlwgt', 'education-num']
Shape            : (48842, 12)
Class distribution: {'<=50K': 37155, '>50K': 11687}

Train: (39073, 12) | Test: (9769, 12)
Feature names: ['age', 'workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']


## 3. Serialization
Three formats from the LIFT / TabLLM literature. `fnlwgt` and `education-num` are already removed from `feature_names`.

| Format | Example |
|---|---|
| `lift` | `age is 39. workclass is State-gov.` |
| `list` | `Feature age: 39, Feature workclass: State-gov.` |
| `compact` | `age=39, workclass=State-gov` |

In [5]:
def serialize_row(row: pd.Series, feature_names: list, fmt: str = "lift") -> str:
    """Convert one tabular row into natural language. Missing values skipped."""
    pairs = []
    for col in feature_names:
        val = row[col]
        if pd.isna(val) or str(val).strip() in ("?", "nan", ""):
            continue
        if isinstance(val, float):
            val = round(val, 4)
        pairs.append((col, val))

    if fmt == "lift":
        return ". ".join(f"{k} is {v}" for k, v in pairs) + "."
    elif fmt == "list":
        return ", ".join(f"Feature {k}: {v}" for k, v in pairs) + "."
    elif fmt == "compact":
        return ", ".join(f"{k}={v}" for k, v in pairs)
    else:
        raise ValueError(f"Unknown format {fmt!r}. Choose: lift | list | compact")


def build_prompt(serialized_row: str, task_prompt: str, use_cot: bool = False) -> str:
    """Assemble full LLM prompt. CoT version requests step-by-step reasoning."""
    if use_cot:
        return (
            f"{serialized_row}\n\n"
            f"Question: {task_prompt}\n"
            f"Think step by step. After reasoning, end with "
            f"'Final Answer: >50K' or 'Final Answer: <=50K'.\n\nReasoning:"
        )
    return f"{serialized_row}\n\nQuestion: {task_prompt}\nAnswer:"


def serialize_dataset(X: pd.DataFrame, feature_names: list,
                      task_prompt: str, fmt: str = "lift",
                      use_cot: bool = False) -> list:
    """Serialize entire DataFrame into LLM-ready prompts."""
    return [
        build_prompt(serialize_row(row, feature_names, fmt), task_prompt, use_cot)
        for _, row in X.iterrows()
    ]


sample_row = X_train.iloc[0]
for fmt in ["lift", "list", "compact"]:
    print(f"\n--- {fmt.upper()} ---")
    print(serialize_row(sample_row, feature_names, fmt=fmt))

print("\n--- DIRECT PROMPT SAMPLE ---")
print(build_prompt(serialize_row(sample_row, feature_names, "lift"),
                   CONFIG["task_prompt"], use_cot=False))

print("\n--- CoT PROMPT SAMPLE ---")
print(build_prompt(serialize_row(sample_row, feature_names, "lift"),
                   CONFIG["task_prompt"], use_cot=True))


--- LIFT ---
age is 37. workclass is Private. education is Bachelors. marital-status is Never-married. occupation is Sales. relationship is Not-in-family. race is White. sex is Female. capital-gain is 0. capital-loss is 0. hours-per-week is 30. native-country is United-States.

--- LIST ---
Feature age: 37, Feature workclass: Private, Feature education: Bachelors, Feature marital-status: Never-married, Feature occupation: Sales, Feature relationship: Not-in-family, Feature race: White, Feature sex: Female, Feature capital-gain: 0, Feature capital-loss: 0, Feature hours-per-week: 30, Feature native-country: United-States.

--- COMPACT ---
age=37, workclass=Private, education=Bachelors, marital-status=Never-married, occupation=Sales, relationship=Not-in-family, race=White, sex=Female, capital-gain=0, capital-loss=0, hours-per-week=30, native-country=United-States

--- DIRECT PROMPT SAMPLE ---
age is 37. workclass is Private. education is Bachelors. marital-status is Never-married. occup

## 4. MLflow Setup
All runs are grouped under the `lift-adult-v2` experiment.

In [14]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

In [15]:
import mlflow
import mlflow.data
from pathlib import Path
from datetime import datetime

MLFLOW_DIR = Path("/kaggle/working/mlruns")
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"file:{MLFLOW_DIR.resolve()}")
mlflow.set_experiment("lift-adult-v2")

DATASET_CONTEXT = {
    "dataset_name"        : "adult_uci_v2",
    "train_rows"          : len(X_train),
    "test_rows"           : len(X_test),
    "n_features"          : X_train.shape[1],
    "dropped_cols"        : "fnlwgt,education-num",
    "label_normalization" : "trailing_dot_stripped",
}

def log_common_context(fmt: str, use_cot: bool):
    params = {
        "model_id"            : CONFIG["model_id"],
        "task_prompt"         : CONFIG["task_prompt"],
        "serial_format"       : fmt,
        "use_cot"             : use_cot,
        "device"              : CONFIG["device"],
        "use_4bit"            : CONFIG["use_4bit"],
        "n_inference_samples" : CONFIG["n_inference_samples"],
        "batch_size"          : CONFIG["batch_size"],
    }
    params.update(DATASET_CONTEXT)
    mlflow.log_params(params)
    mlflow.set_tags({
        "notebook"  : "adult-llm-mlflow-v2.ipynb",
        "platform"  : "kaggle",
        "task_type" : "tabular_text_classification",
        "framework" : "transformers",
        "version"   : "v2-fixed",
    })

print("MLflow ready at:", MLFLOW_DIR.resolve())

2026/06/03 07:59:32 INFO mlflow.tracking.fluent: Experiment with name 'lift-adult-v2' does not exist. Creating a new experiment.


MLflow ready at: /kaggle/working/mlruns


## 5. Load LLM
**FIX**: `padding_side='left'` is mandatory for batched causal LM inference. `dtype=` replaces deprecated `torch_dtype=`.

In [7]:
# Load HuggingFace token from Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets.")
except Exception:
    print("No Kaggle secrets found — set HF_TOKEN env variable manually if needed.")

HF_TOKEN loaded from Kaggle secrets.


In [8]:
def load_model(model_id: str, use_4bit: bool, device: str):
    """Load a HuggingFace causal LM. Returns (tokenizer, model)."""
    print(f"Loading: {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token   # needed for batched generation
    tokenizer.padding_side = "left"             # REQUIRED for causal LM batching

    if use_4bit:
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb_cfg,
            device_map="auto", trust_remote_code=True,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id, dtype=torch.float32,
            device_map={"": device}, trust_remote_code=True,
        )

    model.eval()
    n_params = sum(p.numel() for p in model.parameters()) / 1e9
    print(f"Loaded — {n_params:.2f}B params | device: {device} | 4-bit: {use_4bit}")
    return tokenizer, model


tokenizer, model = load_model(CONFIG["model_id"], CONFIG["use_4bit"], CONFIG["device"])

Loading: Qwen/Qwen2.5-3B-Instruct...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded — 1.70B params | device: cuda | 4-bit: True


## 6. Inference
Greedy decoding (`do_sample=False`) for reproducibility. `max_new_tokens=10` for direct mode is sufficient.

In [16]:
def run_inference(prompts: list, tokenizer, model, device: str,
                  max_new_tokens: int = 10, batch_size: int = 8) -> list:
    """
    Run batched generation. Returns only newly generated tokens per prompt.
    batch_size > 1 gives major GPU speedup; keep at 1 for CPU.
    """
    results, t0 = [], time.time()
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i : i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt",
            padding=True, truncation=True, max_length=512,
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        input_len = inputs["input_ids"].shape[1]
        for ids in output_ids:
            text = tokenizer.decode(ids[input_len:], skip_special_tokens=True).strip()
            results.append(text)

        done = min(i + batch_size, len(prompts))
        if done % max(batch_size * 5, 1) == 0 or done == len(prompts):
            elapsed = time.time() - t0
            speed = done / elapsed if elapsed > 0 else 0
            print(f"  {done}/{len(prompts)} | {elapsed:.1f}s | {speed:.1f} samples/s")

    return results

## 7. Output Parsing
- `parse_prediction` — direct mode: checks first line, then full output
- `parse_prediction_cot` — CoT mode: extracts from `Final Answer:` marker first

In [11]:
LABELS = [">50K", "<=50K"]


def parse_prediction(output: str, labels=None) -> Optional[str]:
    """
    Extract prediction from direct-mode output.
    Checks first line first (most reliable), then full text, then keyword heuristics.
    Returns None for unparseable — never silently inflates accuracy.
    """
    if labels is None:
        labels = LABELS
    first_line = output.split("\n")[0].lower()
    for label in labels:
        if label.lower() in first_line:
            return label
    out_lower = output.lower()
    for label in labels:
        if label.lower() in out_lower:
            return label
    if any(w in out_lower for w in ["yes", "over", "more than", "above", "exceeds"]):
        return ">50K"
    if any(w in out_lower for w in ["no", "under", "less", "at most", "does not", "not more"]):
        return "<=50K"
    return None


def parse_prediction_cot(output: str, labels=None) -> Optional[str]:
    """
    Extract prediction from CoT output.
    Looks for 'Final Answer:' marker; falls back to parse_prediction.
    """
    if labels is None:
        labels = LABELS
    lower = output.lower()
    if "final answer:" in lower:
        answer_part = lower.split("final answer:")[-1]
        for label in labels:
            if label.lower() in answer_part:
                return label
    return parse_prediction(output, labels)


def parse_outputs(raw_outputs: list, use_cot: bool) -> list:
    """Dispatch to correct parser based on CoT flag."""
    parser = parse_prediction_cot if use_cot else parse_prediction
    return [parser(o) for o in raw_outputs]

## 8. Full Evaluation
Computes Accuracy, F1 (macro + per-class), Precision, Recall, ROC-AUC, parse rate. Saves confusion matrix as MLflow artifact.

In [12]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns


def evaluate(results_df: pd.DataFrame, model_id: str, fmt: str, use_cot: bool,
             save_dir: str = "/kaggle/working") -> dict:
    """
    Full evaluation on parseable predictions.
    Logs confusion matrix PNG + all metrics to active MLflow run.
    """
    df = results_df[results_df["predicted"].notna()].copy()
    total, parsed = len(results_df), len(df)
    parse_rate = parsed / total if total > 0 else 0.0

    if parsed == 0:
        print("WARNING: 0 parseable predictions. Check parser and model output.")
        return {"parse_rate": 0.0}

    y_true = df["true_label_norm"]
    y_pred = df["predicted"]
    bin_map = {">50K": 1, "<=50K": 0}
    y_true_bin = y_true.map(bin_map)
    y_pred_bin = y_pred.map(bin_map)

    acc      = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_pos   = f1_score(y_true, y_pred, pos_label=">50K", zero_division=0)
    prec     = precision_score(y_true, y_pred, pos_label=">50K", zero_division=0)
    rec      = recall_score(y_true, y_pred, pos_label=">50K", zero_division=0)
    try:
        roc_auc = roc_auc_score(y_true_bin, y_pred_bin)
    except Exception:
        roc_auc = float("nan")

    cot_tag = "CoT" if use_cot else "Direct"
    print(f"\n{'='*55}")
    print(f" {model_id.split('/')[-1]} | {fmt} | {cot_tag}")
    print(f" {parsed}/{total} parseable ({parse_rate:.1%})")
    print("="*55)
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))
    print(f" ROC-AUC : {roc_auc:.4f}")

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=[">50K", "<=50K"])
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=[">50K","<=50K"], yticklabels=[">50K","<=50K"], ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{model_id.split('/')[-1]} | {fmt} | {cot_tag}")
    plt.tight_layout()
    cm_path = f"{save_dir}/cm_{fmt}_{'cot' if use_cot else 'direct'}.png"
    fig.savefig(cm_path, dpi=150)
    plt.close(fig)
    mlflow.log_artifact(cm_path)

    metrics = {
        "accuracy"     : acc,
        "f1_macro"     : f1_macro,
        "f1_pos_50k"   : f1_pos,
        "precision_pos": prec,
        "recall_pos"   : rec,
        "roc_auc"      : roc_auc,
        "parse_rate"   : parse_rate,
        "n_parseable"  : parsed,
        "n_total"      : total,
    }
    mlflow.log_metrics({k: v for k, v in metrics.items() if not (isinstance(v, float) and np.isnan(v))})
    return metrics

## 9. Ablation: Format × CoT
All 6 combinations: `lift/list/compact` × `direct/cot`.

⚠️ **Runtime estimate on T4 (4-bit, batch=8, n=500)**:
- Direct: ~3–5 min per format (`max_new_tokens=10`)
- CoT: ~20–30 min per format (`max_new_tokens=200`)

In [17]:
all_results = {}  # (fmt, mode) -> metrics dict

n = CONFIG["n_inference_samples"]
X_sample = X_test.iloc[:n].copy()
y_sample  = y_test.iloc[:n]

for fmt in ["lift", "list", "compact"]:
    for use_cot in [False, True]:
        run_key  = (fmt, "cot" if use_cot else "direct")
        max_tok  = CONFIG["max_new_tokens_cot"] if use_cot else CONFIG["max_new_tokens_direct"]
        run_name = (
            f"{CONFIG['model_id'].split('/')[-1]}_{fmt}_"
            f"{'cot' if use_cot else 'direct'}_n{n}_"
            f"{datetime.now().strftime('%Y%m%d-%H%M%S')}"
        )

        print(f"\n{'='*60}")
        print(f" Format: {fmt} | CoT: {use_cot} | max_tokens: {max_tok}")
        print(f"{'='*60}")

        prompts = serialize_dataset(
            X_sample, feature_names,
            task_prompt=CONFIG["task_prompt"],
            fmt=fmt, use_cot=use_cot,
        )

        t_start = time.time()
        with mlflow.start_run(run_name=run_name):
            log_common_context(fmt, use_cot)
            mlflow.log_param("max_new_tokens", max_tok)

            raw_outputs = run_inference(
                prompts, tokenizer, model,
                device=CONFIG["device"],
                max_new_tokens=max_tok,
                batch_size=CONFIG["batch_size"],
            )

            predictions      = parse_outputs(raw_outputs, use_cot)
            true_labels_norm = [normalize_label(l) for l in y_sample.tolist()]

            results_df = pd.DataFrame({
                "prompt"          : [p[:120] + "..." for p in prompts],
                "raw_output"      : raw_outputs,
                "predicted"       : predictions,
                "true_label"      : y_sample.tolist(),
                "true_label_norm" : true_labels_norm,
                "correct"         : [p == t for p, t in zip(predictions, true_labels_norm)],
            })

            mlflow.log_metric("inference_time_s", time.time() - t_start)
            metrics = evaluate(results_df, CONFIG["model_id"], fmt, use_cot)
            all_results[run_key] = metrics

            csv_path = f"/kaggle/working/results_{fmt}_{'cot' if use_cot else 'direct'}.csv"
            results_df.to_csv(csv_path, index=False)
            mlflow.log_artifact(csv_path)

print("\n✅ Ablation complete.")


 Format: lift | CoT: False | max_tokens: 10


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  40/500 | 12.0s | 3.3 samples/s
  80/500 | 22.0s | 3.6 samples/s
  120/500 | 32.1s | 3.7 samples/s
  160/500 | 42.1s | 3.8 samples/s
  200/500 | 52.2s | 3.8 samples/s
  240/500 | 62.2s | 3.9 samples/s
  280/500 | 72.3s | 3.9 samples/s
  320/500 | 82.3s | 3.9 samples/s
  360/500 | 92.4s | 3.9 samples/s
  400/500 | 102.4s | 3.9 samples/s
  440/500 | 112.5s | 3.9 samples/s
  480/500 | 122.5s | 3.9 samples/s
  500/500 | 128.5s | 3.9 samples/s

 Qwen2.5-3B-Instruct | lift | Direct
 500/500 parseable (100.0%)
              precision    recall  f1-score   support

       <=50K     0.8087    0.9949    0.8922       391
        >50K     0.8947    0.1560    0.2656       109

    accuracy                         0.8120       500
   macro avg     0.8517    0.5754    0.5789       500
weighted avg     0.8275    0.8120    0.7556       500

 ROC-AUC : 0.5754

 Format: lift | CoT: True | max_tokens: 200
  40/500 | 205.9s | 0.2 samples/s
  80/500 | 411.1s | 0.2 samples/s
  120/500 | 617.0s | 0.2 samples

## 10. Results Summary Table

In [18]:
rows = []
for (fmt, mode), m in all_results.items():
    rows.append({
        "format"     : fmt,
        "mode"       : mode,
        "accuracy"   : m.get("accuracy", float("nan")),
        "f1_macro"   : m.get("f1_macro", float("nan")),
        "f1_>50K"    : m.get("f1_pos_50k", float("nan")),
        "roc_auc"    : m.get("roc_auc", float("nan")),
        "parse_rate" : m.get("parse_rate", float("nan")),
        "n_parseable": m.get("n_parseable", 0),
    })

summary_df = pd.DataFrame(rows).sort_values(["format", "mode"]).round(4)
display(summary_df)

summary_df.to_csv("/kaggle/working/ablation_summary.csv", index=False)
print("Saved ablation_summary.csv")

,format,mode,accuracy,f1_macro,f1_>50K,roc_auc,parse_rate,n_parseable
5,compact,cot,0.7000,0.5127,0.2105,0.5137,1.000,500
4,compact,direct,0.8038,0.5180,0.1468,0.5387,0.948,474
1,lift,cot,0.7240,0.5979,0.3727,0.5986,1.000,500
0,lift,direct,0.8120,0.5789,0.2656,0.5754,1.000,500
3,list,cot,0.6760,0.5280,0.2636,0.5282,1.000,500
2,list,direct,0.7880,0.4671,0.0536,0.5138,1.000,500


Saved ablation_summary.csv


## 11. Perturbation-Based XAI
Leave-one-out masking: replace each feature with `[UNKNOWN]` and measure accuracy drop.
High drop → model relies heavily on that feature. Analogous to permutation importance.

In [ ]:
N_XAI     = min(100, CONFIG["n_inference_samples"])
XAI_FMT   = "lift"
XAI_COT   = False
XAI_TOK   = CONFIG["max_new_tokens_direct"]

X_xai = X_test.iloc[:N_XAI].copy()
y_xai = [normalize_label(l) for l in y_test.iloc[:N_XAI].tolist()]


def serialize_row_masked(row: pd.Series, feature_names: list,
                         mask_col: str, fmt: str = "lift") -> str:
    """Serialize row with one column replaced by '[UNKNOWN]'."""
    pairs = []
    for col in feature_names:
        if col == mask_col:
            pairs.append((col, "[UNKNOWN]"))
            continue
        val = row[col]
        if pd.isna(val) or str(val).strip() in ("?", "nan", ""):
            continue
        if isinstance(val, float):
            val = round(val, 4)
        pairs.append((col, val))

    if fmt == "lift":
        return ". ".join(f"{k} is {v}" for k, v in pairs) + "."
    elif fmt == "list":
        return ", ".join(f"Feature {k}: {v}" for k, v in pairs) + "."
    return ", ".join(f"{k}={v}" for k, v in pairs)


# Baseline accuracy on XAI subset
baseline_prompts = serialize_dataset(
    X_xai, feature_names,
    task_prompt=CONFIG["task_prompt"],
    fmt=XAI_FMT, use_cot=XAI_COT,
)
baseline_preds = parse_outputs(
    run_inference(baseline_prompts, tokenizer, model,
                  device=CONFIG["device"], max_new_tokens=XAI_TOK,
                  batch_size=CONFIG["batch_size"]),
    use_cot=XAI_COT,
)
valid_pairs = [(t, p) for t, p in zip(y_xai, baseline_preds) if p is not None]
baseline_acc = accuracy_score(*zip(*valid_pairs)) if valid_pairs else 0.0
print(f"XAI baseline accuracy ({N_XAI} samples): {baseline_acc:.4f}")

# Per-feature masking loop
importance_scores = {}
for col in feature_names:
    masked_prompts = [
        build_prompt(
            serialize_row_masked(row, feature_names, mask_col=col, fmt=XAI_FMT),
            CONFIG["task_prompt"], use_cot=XAI_COT,
        )
        for _, row in X_xai.iterrows()
    ]
    masked_preds = parse_outputs(
        run_inference(masked_prompts, tokenizer, model,
                      device=CONFIG["device"], max_new_tokens=XAI_TOK,
                      batch_size=CONFIG["batch_size"]),
        use_cot=XAI_COT,
    )
    vp = [(t, p) for t, p in zip(y_xai, masked_preds) if p is not None]
    masked_acc = accuracy_score(*zip(*vp)) if vp else 0.0
    drop = baseline_acc - masked_acc
    importance_scores[col] = drop
    print(f"  {col:20s}  masked_acc={masked_acc:.4f}  drop={drop:+.4f}")

# Plot
imp_df = (
    pd.DataFrame.from_dict(importance_scores, orient="index", columns=["importance_drop"])
    .sort_values("importance_drop", ascending=False)
)
fig, ax = plt.subplots(figsize=(8, 5))
imp_df["importance_drop"].plot(kind="barh", ax=ax, color="steelblue")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Accuracy drop when feature is masked (↑ = more important)")
ax.set_title(f"Perturbation Feature Importance\n{CONFIG['model_id'].split('/')[-1]} | {XAI_FMT} | Direct")
ax.invert_yaxis()
plt.tight_layout()
xai_path = "/kaggle/working/xai_perturbation_importance.png"
fig.savefig(xai_path, dpi=150)
plt.close(fig)
print(f"\nSaved: {xai_path}")

with mlflow.start_run(run_name=f"xai_perturbation_{XAI_FMT}"):
    log_common_context(XAI_FMT, XAI_COT)
    mlflow.log_param("xai_method", "leave_one_out_masking")
    mlflow.log_param("n_xai_samples", N_XAI)
    mlflow.log_metric("baseline_acc_xai", baseline_acc)
    for col, score in importance_scores.items():
        mlflow.log_metric(f"imp_{col.replace('-', '_')}", score)
    mlflow.log_artifact(xai_path)
    imp_df.to_csv("/kaggle/working/xai_importance.csv")
    mlflow.log_artifact("/kaggle/working/xai_importance.csv")

  40/100 | 10.4s | 3.9 samples/s
  80/100 | 20.4s | 3.9 samples/s
  100/100 | 26.3s | 3.8 samples/s
XAI baseline accuracy (100 samples): 0.8400
  40/100 | 10.2s | 3.9 samples/s
  80/100 | 20.2s | 4.0 samples/s
  100/100 | 26.1s | 3.8 samples/s
  age                   masked_acc=0.8200  drop=+0.0200


## 12. CoT Faithfulness Check
For CoT outputs: does the model mention the features that perturbation analysis says matter most?
**Faithfulness Overlap@K** = fraction of top-K important features mentioned in the reasoning chain.

In [ ]:
def extract_mentioned_features(cot_output: str, feature_names: list) -> list:
    """Return feature names that appear in the CoT reasoning text."""
    lower = cot_output.lower()
    return [f for f in feature_names if f.replace("-", " ").lower() in lower]


def faithfulness_overlap_at_k(cot_outputs: list, feature_names: list,
                               importance_scores: dict, k: int = 3) -> float:
    """
    Average Overlap@K between top-K important features and features mentioned in CoT.
    A high score means the model's reasoning aligns with empirical feature importance.
    """
    top_k = set(sorted(importance_scores, key=importance_scores.get, reverse=True)[:k])
    overlaps = [
        len(top_k & set(extract_mentioned_features(out, feature_names))) / k
        for out in cot_outputs
    ]
    return float(np.mean(overlaps)) if overlaps else 0.0


# Run CoT on the XAI subset
cot_prompts_xai = serialize_dataset(
    X_xai, feature_names,
    task_prompt=CONFIG["task_prompt"],
    fmt="lift", use_cot=True,
)
cot_raw = run_inference(
    cot_prompts_xai, tokenizer, model,
    device=CONFIG["device"],
    max_new_tokens=CONFIG["max_new_tokens_cot"],
    batch_size=CONFIG["batch_size"],
)

for k in [3, 5]:
    faith = faithfulness_overlap_at_k(cot_raw, feature_names, importance_scores, k=k)
    print(f"CoT Faithfulness Overlap@{k} : {faith:.4f}")

print("\n--- Feature mention examples (first 5 CoT outputs) ---")
for i, out in enumerate(cot_raw[:5]):
    mentioned = extract_mentioned_features(out, feature_names)
    print(f"[{i}] Mentioned: {mentioned}")
    print(f"     Output snippet: {out[:200]}\n")

## Summary & Next Steps

### What this notebook covers
- ✅ **Label normalization** — UCI dot-suffix bug fixed, assert 2 classes
- ✅ **Non-semantic features removed** — fnlwgt, education-num
- ✅ **Instruction-tuned model** — Llama-3.2-3B-Instruct
- ✅ **4-bit quantization + batched inference** — production-ready GPU setup
- ✅ **Full metrics** — F1, ROC-AUC, confusion matrix per run
- ✅ **Ablation** — 3 formats × Direct/CoT = 6 MLflow runs
- ✅ **Perturbation XAI** — leave-one-out feature importance
- ✅ **Faithfulness metric** — CoT Overlap@K vs. perturbation importance

### Remaining thesis tasks
- [ ] Cross-notebook comparison: LLM perturbation importance vs. LightGBM SHAP
- [ ] Few-shot prompting experiments (2–5 labeled examples in prompt)
- [ ] Attention-based attribution (extract and aggregate attention weights)
- [ ] Statistical significance: McNemar test for format/CoT comparisons
